### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [6]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [7]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [8]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [9]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [10]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [11]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [12]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [13]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [14]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [15]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [16]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [17]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [18]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [19]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [20]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [21]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [22]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [23]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [24]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [25]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [26]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Consigna 1: Vectorización y similaridad de documentos

Tomamos 5 documentos al azar del conjunto de entrenamiento y medimos similaridad coseno con el resto de los documentos. Analizamos los 5 más similares para evaluar si la similaridad tiene sentido semánticamente.

In [24]:
import random
import numpy as np

# Fijamos semilla para reproducibilidad
random.seed(42)
np.random.seed(42)

# Seleccionamos 5 índices al azar
sample_indices = random.sample(range(len(newsgroups_train.data)), 5)
print('Índices seleccionados:', sample_indices)


Índices seleccionados: [10476, 1824, 409, 4506, 4012]


In [25]:
for idx in sample_indices:
    # Texto y etiqueta del documento base
    clase_doc = newsgroups_train.target_names[y_train[idx]]
    print('=' * 70)
    print(f'Documento #{idx} | Clase: {clase_doc}')
    print('-' * 70)
    print(newsgroups_train.data[idx][:300], '...')
    print()

    # Similaridad coseno con todos los documentos de train
    cossim = cosine_similarity(X_train[idx], X_train)[0]

    # Los 5 más similares (excluimos el propio documento, índice 0)
    most_sim = np.argsort(cossim)[::-1][1:6]

    print(f'  5 documentos más similares:')
    for rank, sim_idx in enumerate(most_sim, 1):
        clase_sim = newsgroups_train.target_names[y_train[sim_idx]]
        print(f'  {rank}. Doc #{sim_idx} | Clase: {clase_sim} | Similaridad: {cossim[sim_idx]:.4f}')
    print()


Documento #10476 | Clase: rec.sport.hockey
----------------------------------------------------------------------
This is a general question for US readers:

How extensive is the playoff coverage down there?  In Canada, it is almost
impossible not to watch a series on TV (ie the only two series I have not had
an opportunity to watch this year are Wash-NYI and Chi-Stl, the latter because
I'm in the wrong time zo ...

  5 documentos más similares:
  1. Doc #5064 | Clase: rec.sport.hockey | Similaridad: 0.2250
  2. Doc #9623 | Clase: talk.politics.mideast | Similaridad: 0.2174
  3. Doc #10575 | Clase: sci.crypt | Similaridad: 0.2164
  4. Doc #10836 | Clase: alt.atheism | Similaridad: 0.2126
  5. Doc #2350 | Clase: sci.crypt | Similaridad: 0.2111

Documento #1824 | Clase: comp.sys.mac.hardware
----------------------------------------------------------------------


	I think this kind of comparison is pretty useless in general.  The
processor is only good when a good computer is designed ar

### Análisis Consigna 1

Los resultados muestran un comportamiento heterogéneo que depende fuertemente del contenido y extensión de cada documento.

Los documentos #1824 (comp.sys.mac.hardware) y #409 (comp.graphics) presentan los 5 vecinos más similares pertenecientes exactamente a su misma clase, con similaridades relativamente altas (hasta 0.35). Esto indica que estos textos contienen vocabulario técnico específico y suficiente que permite a TF-IDF discriminarlos correctamente.

En cambio, #10476 (rec.sport.hockey) obtiene solo 1 vecino de su clase entre los 5 más similares, y las similaridades son muy bajas (todas alrededor de 0.21). El texto es corto y usa vocabulario genérico ("watch", "series", "TV"), lo que genera vecinos de clases completamente dispares como sci.crypt o alt.atheism. Esto ilustra una limitación clave de TF-IDF: documentos breves o con vocabulario poco especializado no se representan de forma discriminativa.

En #4506 (rec.autos) y #4012 (rec.sport.hockey) muestran casos intermedios: la mayoría de vecinos son de la clase correcta o clases relacionadas (motorcycles es temáticamente cercana a autos; baseball a hockey), pero aparecen clases sin relación aparente (comp.sys.mac.hardware, soc.religion.christian) probablemente por el escaso contenido de esos documentos.

En síntesis, TF-IDF captura bien la similitud cuando los documentos son suficientemente largos y con vocabulario temático distintivo. Documentos cortos o con lenguaje cotidiano producen representaciones poco informativas y similaridades bajas con resultados poco confiables.

## Consigna 2: Clasificador por prototipos (zero-shot)

Clasificamos cada documento de test comparándolo con todos los documentos de entrenamiento y asignando la clase del documento de train con mayor similaridad coseno. Este enfoque se denomina *1-Nearest Neighbor* o clasificador por prototipos.

In [27]:
# Vectorizamos test con el vectorizador ya ajustado en train
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

# Clasificador por prototipos: 1-Nearest Neighbor por similaridad coseno
y_pred_proto = []

for i in range(X_test.shape[0]):
    cossim = cosine_similarity(X_test[i], X_train)[0]
    nn_idx = np.argmax(cossim)
    y_pred_proto.append(y_train[nn_idx])

y_pred_proto = np.array(y_pred_proto)

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-Score Macro (clasificador por prototipos): {f1_proto:.4f}')

F1-Score Macro (clasificador por prototipos): 0.5050


### Análisis Consigna 2

El clasificador por prototipos obtuvo un F1-Score Macro de 0.5050. Es un enfoque no paramétrico que no requiere entrenamiento explícito: simplemente recupera el documento de train más similar al de test y le asigna su etiqueta.

Este resultado es razonable considerando la simplicidad del método: sin aprender ninguna distribución de probabilidad, logra clasificar correctamente alrededor de la mitad de los documentos en un problema de 20 clases (donde el azar daría ~0.05). Esto habla bien de la capacidad discriminativa de la representación TF-IDF.

Como ventaja, es completamente interpretable: siempre se puede identificar qué documento de train motivó cada predicción. Como desventaja, es computacionalmente costoso en inferencia (O(n_train) por cada predicción) y no aprovecha información estadística global del corpus, lo que limita su desempeño.

Se espera que Naïve Bayes supere este 0.5050, ya que agrega sobre toda la distribución de términos por clase en lugar de depender de un único documento vecino, siendo más robusto ante documentos cortos o con vocabulario poco representativo (como los que vimos en la Consigna 1).

## Consigna 3: Modelos Naïve Bayes – maximizar F1-Score Macro

Probamos distintas configuraciones de vectorizador (TF-IDF y Count) con sus parámetros más relevantes, y comparamos MultinomialNB vs ComplementNB. **No se modifica `ngram_range`.**

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
from itertools import product

results = []

# Configuraciones a explorar
vectorizer_configs = [
    # (nombre, clase, kwargs)
    ('TF-IDF default',        TfidfVectorizer,  {}),
    ('TF-IDF max_df=0.5',     TfidfVectorizer,  {'max_df': 0.5}),
    ('TF-IDF min_df=2',       TfidfVectorizer,  {'min_df': 2}),
    ('TF-IDF sublinear_tf',   TfidfVectorizer,  {'sublinear_tf': True}),
    ('TF-IDF sublin+min_df2', TfidfVectorizer,  {'sublinear_tf': True, 'min_df': 2}),
    ('Count default',         CountVectorizer,  {}),
    ('Count min_df=2',        CountVectorizer,  {'min_df': 2}),
    ('Count max_df=0.5',      CountVectorizer,  {'max_df': 0.5}),
]

model_configs = [
    ('MultinomialNB alpha=1.0', MultinomialNB(alpha=1.0)),
    ('MultinomialNB alpha=0.1', MultinomialNB(alpha=0.1)),
    ('MultinomialNB alpha=0.5', MultinomialNB(alpha=0.5)),
    ('ComplementNB  alpha=1.0', ComplementNB(alpha=1.0)),
    ('ComplementNB  alpha=0.1', ComplementNB(alpha=0.1)),
    ('ComplementNB  alpha=0.5', ComplementNB(alpha=0.5)),
]

for vect_name, VectClass, vect_kwargs in vectorizer_configs:
    vect = VectClass(**vect_kwargs)
    X_tr = vect.fit_transform(newsgroups_train.data)
    X_te = vect.transform(newsgroups_test.data)

    for clf_name, clf_obj in model_configs:
        clf_obj.fit(X_tr, y_train)
        preds = clf_obj.predict(X_te)
        f1 = f1_score(y_test, preds, average='macro')
        results.append({'Vectorizador': vect_name, 'Modelo': clf_name, 'F1-Macro': f1})

# Ordenar por F1 descendente
results.sort(key=lambda x: x['F1-Macro'], reverse=True)

print(f'{'Vectorizador':<26} {'Modelo':<28} F1-Macro')
print('-' * 70)
for r in results[:15]:  # Top 15
    print(f"{r['Vectorizador']:<26} {r['Modelo']:<28} {r['F1-Macro']:.4f}")


Vectorizador               Modelo                       F1-Macro
----------------------------------------------------------------------
TF-IDF min_df=2            ComplementNB  alpha=0.5      0.6980
TF-IDF sublinear_tf        ComplementNB  alpha=0.5      0.6968
TF-IDF sublin+min_df2      ComplementNB  alpha=0.5      0.6964
TF-IDF max_df=0.5          ComplementNB  alpha=0.5      0.6962
TF-IDF default             ComplementNB  alpha=0.5      0.6961
TF-IDF default             ComplementNB  alpha=0.1      0.6954
TF-IDF max_df=0.5          ComplementNB  alpha=0.1      0.6948
TF-IDF min_df=2            ComplementNB  alpha=1.0      0.6935
TF-IDF sublin+min_df2      ComplementNB  alpha=1.0      0.6932
TF-IDF max_df=0.5          ComplementNB  alpha=1.0      0.6930
TF-IDF default             ComplementNB  alpha=1.0      0.6930
TF-IDF sublinear_tf        ComplementNB  alpha=0.1      0.6927
TF-IDF sublinear_tf        ComplementNB  alpha=1.0      0.6920
TF-IDF min_df=2            ComplementNB  alph

In [29]:
# Mejor configuración encontrada
best = results[0]
print(f"Mejor configuración:")
print(f"  Vectorizador : {best['Vectorizador']}")
print(f"  Modelo       : {best['Modelo']}")
print(f"  F1-Macro     : {best['F1-Macro']:.4f}")


Mejor configuración:
  Vectorizador : TF-IDF min_df=2
  Modelo       : ComplementNB  alpha=0.5
  F1-Macro     : 0.6980


### Análisis Consigna 3

La mejor configuración encontrada fue TF-IDF con min_df=2 + ComplementNB con alpha=0.5, alcanzando un F1-Score Macro de 0.6980, una mejora considerable sobre el clasificador por prototipos (0.5050).

Varios patrones claros emergen del ranking:
ComplementNB domina completamente el top 15. Ninguna configuración con MultinomialNB aparece entre los mejores resultados. Esto se explica porque ComplementNB estima la probabilidad de que un documento no pertenezca a cada clase, lo que lo hace más robusto en datasets con cierto desbalance entre las 20 categorías de newsgroups.

El vectorizador TF-IDF supera a CountVectorizer en todas las posiciones del top. TF-IDF penaliza términos muy frecuentes en el corpus (como stopwords que no se filtraron), lo que produce representaciones más discriminativas que el conteo crudo.

alpha=0.5 es el valor óptimo de suavización, superando tanto a alpha=1.0 (suavización estándar de Laplace, demasiado conservadora) como a alpha=0.1 (poca suavización, más sensible a términos raros). El valor intermedio balancea mejor ambos extremos.

min_df=2 da la mejor combinación individual. Eliminar términos que aparecen en un solo documento reduce ruido (errores tipográficos, términos únicos sin poder discriminativo) sin perder vocabulario relevante. Los demás parámetros como max_df=0.5 o sublinear_tf muestran mejoras marginales respecto al default.

## Consigna 4: Matriz término-documento y similaridad entre palabras

Transponemos la matriz documento-término para obtener la **matriz término-documento**, donde cada fila es un vector de contexto de una palabra (en qué documentos aparece y con qué peso). Esto nos permite medir similaridad entre palabras de forma análoga a embeddings distribuidos simples.

Seleccionamos **5 palabras manualmente** eligiendo términos temáticos y unívocos del dataset 20 newsgroups.

In [30]:
# Usamos el mejor vectorizador encontrado en la consigna 3
# TF-IDF con min_df=2 + ComplementNB alpha=0.5 → F1-Macro: 0.6980
vect_c4 = TfidfVectorizer(min_df=2)
X_train_c4 = vect_c4.fit_transform(newsgroups_train.data)

# Transponer: filas = términos, columnas = documentos
X_term_doc = X_train_c4.T  # shape: (vocab_size, n_docs)

vocab_c4 = vect_c4.get_feature_names_out()
vocab_to_idx = vect_c4.vocabulary_

print(f'Forma de la matriz término-documento: {X_term_doc.shape}')
print(f'Vocabulario total: {len(vocab_c4)} términos')


Forma de la matriz término-documento: (39423, 11314)
Vocabulario total: 39423 términos


In [31]:
# Palabras seleccionadas manualmente — temáticas y representativas del dataset
words_of_interest = ['space', 'hockey', 'encryption', 'religion', 'medicine']

for word in words_of_interest:
    if word not in vocab_to_idx:
        print(f"'{word}' no está en el vocabulario")
        continue

    w_idx = vocab_to_idx[word]
    # Vector del término: su contexto en los documentos
    w_vec = X_term_doc[w_idx]  # sparse row

    # Similaridad con todos los demás términos
    sims_words = cosine_similarity(w_vec, X_term_doc)[0]

    # Los 5 más similares (excluimos la palabra misma)
    top5_idx = np.argsort(sims_words)[::-1]
    top5_filtered = [i for i in top5_idx if i != w_idx][:5]

    print(f"Palabra: '{word}'")
    for rank, t_idx in enumerate(top5_filtered, 1):
        print(f"  {rank}. '{vocab_c4[t_idx]}' (similaridad: {sims_words[t_idx]:.4f})")
    print()


Palabra: 'space'
  1. 'nasa' (similaridad: 0.3305)
  2. 'seds' (similaridad: 0.2933)
  3. 'shuttle' (similaridad: 0.2881)
  4. 'enfant' (similaridad: 0.2775)
  5. 'seti' (similaridad: 0.2460)

Palabra: 'hockey'
  1. 'ncaa' (similaridad: 0.2698)
  2. 'nhl' (similaridad: 0.2612)
  3. 'affiliates' (similaridad: 0.2441)
  4. 'sportschannel' (similaridad: 0.2182)
  5. 'boondock' (similaridad: 0.2054)

Palabra: 'encryption'
  1. 'microcircuit' (similaridad: 0.3349)
  2. 'accommodates' (similaridad: 0.3329)
  3. 'nistnews' (similaridad: 0.3329)
  4. 'pitted' (similaridad: 0.3329)
  5. 'torrance' (similaridad: 0.3329)

Palabra: 'religion'
  1. 'religious' (similaridad: 0.2373)
  2. 'religions' (similaridad: 0.2113)
  3. 'categorized' (similaridad: 0.1987)
  4. 'crusades' (similaridad: 0.1943)
  5. 'christianity' (similaridad: 0.1726)

Palabra: 'medicine'
  1. 'strengthens' (similaridad: 0.3544)
  2. 'dislikes' (similaridad: 0.3351)
  3. 'nearer' (similaridad: 0.2933)
  4. 'foremost' (similarid

### Análisis Consigna 4

Los resultados muestran comportamientos muy distintos según la palabra, lo que permite sacar conclusiones interesantes sobre las limitaciones de este enfoque.

'space' es el caso más exitoso: sus palabras similares (nasa, shuttle, seti, seds) son todas términos directamente relacionados con exploración espacial y astronomía. Las similaridades son moderadas (~0.29-0.33) pero semánticamente muy coherentes. TF-IDF captura bien este campo temático porque tiene vocabulario muy específico y concentrado en documentos del grupo sci.space.

'hockey' tiene resultados parcialmente coherentes: nhl (liga de hockey) y ncaa (organismo deportivo universitario) tienen sentido. Sin embargo, affiliates, sportschannel y boondock son términos más relacionados con la distribución televisiva de partidos que con el deporte en sí, lo que refleja que los documentos de hockey en este corpus discutían seguramente mucho sobre la cobertura mediática (consistente con el documento #10476 visto en la consigna 1).

'encryption' y 'medicine' muestran el comportamiento menos alentador: sus palabras similares son términos completamente inconexos y sin interpretación semántica clara (microcircuit, nistnews, strengthens, dislikes). Esto ocurre porque estas palabras aparecen en pocos documentos muy específicos, y con min_df=2 apenas superan el umbral mínimo. Sus vectores de contexto son muy dispersos y la similaridad coseno se ve dominada por co-ocurrencias accidentales en esos pocos documentos.

'religion' es un caso intermedio: religious, religions y christianity son claramente relacionadas, pero categorized y crusades son más ruidosas. Las similaridades bajas (máximo 0.24) indican que "religion" aparece en contextos variados del corpus, diluyendo su vector.

En síntesis, este enfoque funciona bien para términos con vocabulario temático denso y específico (como space), pero falla con palabras que aparecen en contextos heterogéneos o en pocos documentos. Para capturar similaridad semántica real se necesitarían embeddings neuronales por ejemplo, que consideren contexto local en lugar de co-ocurrencia documental global.